# Query Generation from LongEval 2026 Snapshot Parquet

This notebook outlines the process of loading raw data from a Parquet file and formatting it into a structured sample query dataset

In [1]:
!pip install pandas pyarrow sentence-transformers rank-bm25 scikit-learn

In [2]:
from google.colab import drive
import pyarrow.parquet as pq
drive.mount('/content/drive')

path = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/longeval_documents.parquet"
pf = pq.ParquetFile(path)
print("Num row groups:", pf.num_row_groups)
print("Schema:", pf.schema)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Num row groups: 8
Schema: <pyarrow._parquet.ParquetSchema object at 0x7fb8e9cc2080>
required group field_id=-1 duckdb_schema {
  optional binary field_id=-1 doc_id (String);
  optional binary field_id=-1 original_text (String);
  optional binary field_id=-1 metadata (JSON);
}



In [5]:
import pandas as pd
pieces = []
rows_per_group = 5  # small and safe

for rg in range(min(3, pf.num_row_groups)):   # first 3 row groups
    t = pf.read_row_group(rg, columns=["doc_id", "original_text", "metadata"])
    pieces.append(t.slice(0, rows_per_group).to_pandas())

df_sample = pd.concat(pieces, ignore_index=True)
print(df_sample.shape)
df_sample.head(3)

(15, 3)


,doc_id,original_text,metadata
0,8724,Two new species of Pterostichus Bonelli subgen...,"{""arxivId"":null,""authors"":[{""name"":""Bergdahl,J..."
1,8725,When to sample in an inaccessible landscape: a...,"{""arxivId"":null,""authors"":[{""name"":""Harry, Ing..."
2,8820,Twenty-four new species of Polycentropus (Tric...,"{""arxivId"":null,""authors"":[{""name"":""Steven Ham..."


In [6]:
import json

for i in range(len(df_sample)):
    print(f"\n--- DOC {i} ---")
    print("doc_id:", df_sample.loc[i, "doc_id"])
    text = df_sample.loc[i, "original_text"]
    print(text[:2000])
    print("\n" + "="*100)


--- DOC 0 ---
doc_id: 8724
Two new species of Pterostichus Bonelli subgenus Pseudoferonina Ball (Coleoptera, Carabidae, Pterostichini) from the mountains of central Idaho, U.S.A.

Two new species of Pterostichus Bonelli subgenus Pseudoferonina Ball, are described from the mountains of central Idaho: Pterostichus bousqueti Bergdahl [type locality = small tributaries of South Fork of Payette River watershed, ca. 1170 m (3840 ft), 44.0675°/-115.6822°, near Lowman, Salmon River Mountains, Boise County, Idaho, U.S.A.] and Pterostichus lolo Bergdahl [type locality = Cottonwood/Orogrande Creek, ca. 870 m (2850 ft), 46.5528°/-115.5522°, North Fork of Clearwater River watershed, Clearwater Mountains, near Bungalow, Clearwater County, Idaho, U.S.A.]. Males of P. bousqueti and P. lolo are easily distinguished from each other and the seven previously described Pseudoferonina species by the form of the median lobe of the aedeagus, and from most individuals of the other species of Pseudoferonina in

## Parse Metadata into usable fields

In [7]:
def parse_meta(x):
    if isinstance(x, str):
        return json.loads(x)
    return x if isinstance(x, dict) else {}

for i in range(len(df_sample)):
    meta = parse_meta(df_sample.loc[i, "metadata"])
    print(f"\n--- META {i} ---")
    print("publishedDate:", meta.get("publishedDate"))
    print("createdDate:", meta.get("createdDate"))
    authors = meta.get("authors", [])
    print("authors:", [a.get("name") for a in authors[:5]])


--- META 0 ---
publishedDate: 2011-01-01T00:00:00
createdDate: 2014-05-06T17:20:01
authors: ['Bergdahl,James', 'Kavanaugh,David']

--- META 1 ---
publishedDate: 2011-01-01T00:00:00
createdDate: 2014-05-06T17:20:01
authors: ['Harry, Ingmar', 'Drees, Claudia', 'Hofer, Hubert', 'Assmann, Thorsten']

--- META 2 ---
publishedDate: 2011-01-01T00:00:00
createdDate: 2014-05-06T17:20:01
authors: ['Steven Hamilton', 'Ralph Holzenthal']

--- META 3 ---
publishedDate: 2011-01-01T00:00:00
createdDate: 2013-07-12T18:08:02
authors: ['Menzel,Lena']

--- META 4 ---
publishedDate: 2011-01-01T00:00:00
createdDate: 2014-05-06T17:20:01
authors: ['Jia, Fenglong', 'van Vondel, Bernhard']

--- META 5 ---
publishedDate: 2014-01-01T00:00:00
createdDate: 2014-10-24T19:29:19
authors: ['Ho, Luis C.', 'Shen, Yue']

--- META 6 ---
publishedDate: 2014-09-23T01:00:00
createdDate: 2014-10-24T19:29:38
authors: ['Bao, Jianhai', 'Shao, Jinghai', 'Yuan, Chenggui']

--- META 7 ---
publishedDate: 2014-09-08T01:00:00
created

### Format Queries
Now we will map the existing columns to the required format: `query_id`, `query`, and any additional metadata.

In [8]:
template_categories = {
    "general_findings": [
        "What are the main findings described in the provided documents regarding {}?",
        "What conclusions do the retrieved documents draw about {}?",
        "What insights do the provided studies offer about {}?",
        "How do the candidate documents summarize the current understanding of {}?",
        "What key results are presented in the provided documents concerning {}?",
    ],

    "mechanisms_processes": [
        "What mechanisms related to {} are described in the provided documents?",
        "How do the candidate documents explain the processes underlying {}?",
        "What biological, physical, or computational mechanisms are discussed in relation to {}?",
        "What processes or pathways associated with {} are identified in the provided studies?",
        "How do the retrieved documents describe the operation or functioning of {}?",
    ],

    "traits_properties_characteristics": [
        "What main characteristics of {} are discussed in the provided documents?",
        "How do the candidate documents characterize {}?",
        "What traits, properties, or defining features of {} are identified in the provided studies?",
        "What notable structural or functional characteristics of {} are reported?",
        "What distinguishing attributes of {} are described in the retrieved documents?",
    ],

    "variation_differences_comparison": [
        "What differences or variations related to {} are discussed in the provided documents?",
        "How do the candidate documents compare different aspects of {}?",
        "What distinctions are identified in the provided studies concerning {}?",
        "What types of variation in {} are reported by the retrieved documents?",
        "How do the provided documents differentiate among forms, types, or behaviors of {}?",
    ],

    "challenges_limitations_tradeoffs": [
        "What challenges associated with {} are described in the provided documents?",
        "What limitations or difficulties related to {} are identified in the candidate documents?",
        "What tradeoffs involving {} are discussed in the provided studies?",
        "How do the retrieved documents describe the main obstacles or constraints affecting {}?",
        "What problems or unresolved issues concerning {} are highlighted in the provided documents?",
    ],

    "methods_approaches_techniques": [
        "What methods or approaches related to {} are described in the provided documents?",
        "How do the candidate documents study or analyze {}?",
        "What techniques are used in the provided studies to investigate {}?",
        "What methodological approaches to {} are discussed in the retrieved documents?",
        "How do the provided documents describe the procedures or strategies used for {}?",
    ],

    "evidence_observations_results": [
        "What evidence do the provided documents give about {}?",
        "What observations related to {} are reported in the candidate documents?",
        "What experimental or empirical results concerning {} are described in the provided studies?",
        "What supporting evidence is presented in the retrieved documents for {}?",
        "What results do the provided documents report regarding {}?",
    ],

    "causes_factors_drivers": [
        "What factors influencing {} are identified in the provided documents?",
        "What causes or drivers of {} are discussed in the candidate documents?",
        "How do the provided studies explain the factors affecting {}?",
        "What variables associated with {} are reported in the retrieved documents?",
        "What contributing influences on {} are described in the provided documents?",
    ],

    "implications_applications_significance": [
        "What implications of {} are discussed in the provided documents?",
        "How do the candidate documents describe the significance of {}?",
        "What applications or broader impacts of {} are identified in the provided studies?",
        "Why is {} considered important according to the retrieved documents?",
        "What practical or theoretical relevance of {} is described in the provided documents?",
    ],

    "temporal_development_change": [
        "How is the development of {} described in the provided documents?",
        "What changes in {} are reported by the candidate documents?",
        "How do the provided studies describe the progression or evolution of {}?",
        "What stages, trends, or temporal patterns related to {} are discussed in the retrieved documents?",
        "How does {} change over time according to the provided documents?",
    ],
}

In [20]:
short_template_categories = {
    "general": [
        "What are the main findings on {}?",
        "What do the provided documents say about {}?",
        "What is reported about {}?",
        "What are the key results on {}?",
    ],

    "mechanisms": [
        "What mechanisms are described for {}?",
        "How is {} explained in the provided documents?",
        "What processes underlie {}?",
    ],

    "characteristics": [
        "What characteristics of {} are described?",
        "What features of {} are discussed?",
        "How is {} characterized?",
    ],

    "variation": [
        "What variation in {} is reported?",
        "What differences in {} are discussed?",
        "How does {} vary across the documents?",
    ],

    "challenges": [
        "What challenges are discussed for {}?",
        "What limitations of {} are identified?",
        "What tradeoffs are reported for {}?",
    ],

    "methods": [
        "What methods are used to study {}?",
        "How is {} analyzed in the provided documents?",
        "What approaches are described for {}?",
    ],

    "evidence": [
        "What evidence is presented for {}?",
        "What observations are reported about {}?",
        "What results support {}?",
    ],

    "factors": [
        "What factors affect {}?",
        "What influences on {} are identified?",
        "What drives {} according to the documents?",
    ],

    "implications": [
        "What is the significance of {}?",
        "What implications of {} are discussed?",
        "Why is {} important in the provided documents?",
    ],

    "development": [
        "How does {} develop over time?",
        "What stages of {} are described?",
        "How is the progression of {} discussed?",
    ],
}

In [9]:
def infer_categories_from_text(text):
    text = (text or "").lower()
    cats = []

    if any(w in text for w in ["method", "approach", "algorithm", "procedure", "model"]):
        cats.append("methods_approaches_techniques")
    if any(w in text for w in ["mechanism", "pathway", "process", "regulation"]):
        cats.append("mechanisms_processes")
    if any(w in text for w in ["difference", "variation", "dimorphism", "compare"]):
        cats.append("variation_differences_comparison")
    if any(w in text for w in ["challenge", "limitation", "tradeoff", "constraint"]):
        cats.append("challenges_limitations_tradeoffs")
    if any(w in text for w in ["factor", "driver", "cause", "determinant"]):
        cats.append("causes_factors_drivers")
    if any(w in text for w in ["result", "observation", "evidence", "finding"]):
        cats.append("evidence_observations_results")
    if any(w in text for w in ["development", "stage", "evolution", "temporal", "over time"]):
        cats.append("temporal_development_change")
    if any(w in text for w in ["trait", "feature", "characteristic", "property", "structure"]):
        cats.append("traits_properties_characteristics")
    if any(w in text for w in ["application", "significance", "importance", "impact"]):
        cats.append("implications_applications_significance")

    if not cats:
        cats = ["general_findings"]

    return cats

In [22]:
import random
import re
import pandas as pd

def normalize_ws(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()

def shorten_topic(text, max_words=12, max_chars=90):
    text = normalize_ws(text)
    if not text:
        return "the topic"

    # remove abstract-like continuation after first sentence if present
    first_sentence = re.split(r'(?<=[.?!])\s+', text, maxsplit=1)[0]
    candidate = first_sentence

    # if the first sentence is still too long, cut by punctuation
    candidate = re.split(r"[:;]", candidate)[0]

    # if still too long, trim by words
    words = candidate.split()
    if len(words) > max_words:
        candidate = " ".join(words[:max_words])

    # final char trim
    if len(candidate) > max_chars:
        candidate = candidate[:max_chars].rsplit(" ", 1)[0]

    # clean trailing punctuation
    candidate = re.sub(r"[\s,;:.?-]+$", "", candidate)

    return candidate if len(candidate) >= 8 else text[:max_chars].rsplit(" ", 1)[0]

def clean_topic(topic, max_len=180):
    topic = normalize_ws(topic)

    # remove trailing punctuation junk
    topic = re.sub(r"\s*[,:;.-]+\s*$", "", topic)

    # trim cleanly
    if len(topic) > max_len:
        cut = topic[:max_len]
        last_space = cut.rfind(" ")
        if last_space > 60:
            topic = cut[:last_space]
        else:
            topic = cut

    return topic.strip()

def extract_topic_from_text(text, max_topic_chars=180):
    """
    Heuristic: use the beginning of original_text as the main topic/title-like region.
    Since your data seems to start with title followed by abstract-style text,
    this usually gives a decent topic anchor.
    """
    text = normalize_ws(text)
    if not text:
        return "the topic discussed in the provided documents"

    # Try to stop at a reasonable sentence-ish boundary
    m = re.search(r"[.?!]", text[:250])
    if m and m.start() > 40:
        topic = text[:m.start()]
    else:
        topic = text[:max_topic_chars]

    topic = clean_topic(topic, max_len=max_topic_chars)

    # fallback
    if len(topic) < 15:
        topic = clean_topic(text[:max_topic_chars], max_len=max_topic_chars)

    return topic if topic else "the topic discussed in the provided documents"

def choose_template_category(source_text, preferred_category=None):
    if preferred_category is not None and preferred_category in short_template_categories:
        return preferred_category

    cats = infer_categories_from_text(source_text)
    cats = [c for c in cats if c in template_categories]

    if not cats:
        cats = ["general_findings"]

    return random.choice(cats)

def make_query_from_text(text, preferred_category=None, return_parts=False):
    """
    Main helper:
    - extracts a topic from original_text
    - infers a template category
    - samples a template
    - returns a synthetic Task 4-style query
    """
    topic = extract_topic_from_text(text)
    category = choose_template_category(text, preferred_category=preferred_category)
    template = random.choice(short_template_categories[category])
    query = template.format(topic)

    if return_parts:
        return {
            "topic": topic,
            "category": category,
            "template": template,
            "query": query,
        }

    return query

def choose_short_category(source_text, preferred_category=None):
    if preferred_category is not None and preferred_category in template_categories:
        return preferred_category

    inferred = infer_categories_from_text(source_text)

    mapping = {
        "general_findings": "general",
        "mechanisms_processes": "mechanisms",
        "traits_properties_characteristics": "characteristics",
        "variation_differences_comparison": "variation",
        "challenges_limitations_tradeoffs": "challenges",
        "methods_approaches_techniques": "methods",
        "evidence_observations_results": "evidence",
        "causes_factors_drivers": "factors",
        "implications_applications_significance": "implications",
        "temporal_development_change": "development",
    }

    short_cats = [mapping[c] for c in inferred if c in mapping]
    if not short_cats:
        short_cats = ["general"]

    return random.choice(short_cats)

def make_short_query_from_text(text, preferred_category=None, return_parts=False):
    topic = shorten_topic(text)
    category = choose_short_category(text, preferred_category=preferred_category)
    template = random.choice(short_template_categories[category])
    query = template.format(topic)

    # final safety trim to roughly 25 words
    words = query.split()
    if len(words) > 25:
        query = " ".join(words[:25]).rstrip(",;:.") + "?"

    if return_parts:
        return {
            "topic": topic,
            "category": category,
            "template": template,
            "query": query,
        }
    return query

In [23]:
def generate_short_synthetic_queries(df, n=10, seed=42):
    random.seed(seed)
    n = min(n, len(df))
    sample_idx = random.sample(range(len(df)), k=n)

    rows = []
    for j, idx in enumerate(sample_idx, start=1):
        text = df.loc[idx, "original_text"]
        doc_id = df.loc[idx, "doc_id"]
        out = make_short_query_from_text(text, return_parts=True)

        rows.append({
            "query_id": f"{j:05d}",
            "anchor_doc_id": doc_id,
            "topic": out["topic"],
            "category": out["category"],
            "query": out["query"],
        })

    return pd.DataFrame(rows)

In [24]:
queries_df = generate_short_synthetic_queries(df_sample, n=10, seed=42)
queries_df[["query_id", "anchor_doc_id", "category", "query"]]

,query_id,anchor_doc_id,category,query
0,00001,70416404,methods,What approaches are described for INVESTIGACIÓ...
1,00002,8725,variation,What variation in When to sample in an inacces...
2,00003,8724,characteristics,What characteristics of Two new species of Pte...
3,00004,70416406,methods,What methods are used to study LEY DE 1903 PRE...
4,00005,8977,general,What do the provided documents say about Annot...
5,00006,8934,development,How is the progression of First descriptions o...
6,00007,17223685,implications,Why is Existence of special primary decomposit...
7,00008,8820,general,What do the provided documents say about Twent...
8,00009,17223640,characteristics,How is The diversity of quasars unified by acc...
9,00010,70416407,evidence,What evidence is presented for DOS ESTUDIOS SO...
